# Big Data Bowl 2027 — Cross-Position Injury Risk & the Combine

**Question:** across position groups, do injured players differ from healthy ones in how they move in games, and do Combine results (timed or tracked) capture those differences?

This notebook runs one pipeline on four position families and writes **one summary row per pre-specified test**. The tests were fixed in the findings doc on Nov 15, 2026, before the receiver and front-seven data were examined.

| Family | Positions | Measure set | Role |
|---|---|---|---|
| Defensive backs | CB, SS, FS | Cutting | Discovery (tests found here) |
| Receivers | WR, TE | Cutting | **Replication** of the DB tests |
| Offensive line | T, G, C | Burst | Discovery |
| Front seven | DE, OLB, DT, NT | Burst | **Replication** of the OL tests |

**Measure sets.** *Cutting* (from `bdb2027_db_injury_risk.ipynb`): top speed, acceleration, deceleration, turn rate, speed kept through cuts, cuts per play. *Burst* (from `bdb2027_ol_injury_risk.ipynb`): get-off, first-step acceleration, stop power, top speed, effort per rep.

**Sections:** 0 setup · 1 injury definition · 2 game measures · 3 pre-specified tests · 4 Combine comparison · 5 run all families · 6 cross-position summary.

**Data:** reads `../data/` (run from `Ben_EDA/`). Rep-level measures are cached to `outputs/cross_position/` after the first run, since loading tracking for each family takes a while. If `../data/` isn't found, it runs on synthetic data from `synthetic_cross.py` (needs `synthetic_db.py` and `synthetic_ol.py` in the same folder).

## 0. Setup

In [ ]:
from pathlib import Path
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

DATA_DIR = Path("../data")
OUT_DIR = Path("outputs/cross_position")
TRACKING_GLOB = "game_tracking_*.csv"
USE_SYNTHETIC = not (DATA_DIR / "games.csv").exists()
USE_CACHE = not USE_SYNTHETIC
SNAP_EVENTS = {"ball_snap", "snap_direct", "autoevent_ballsnap"}

# ---- position families ----
FAMILIES = {
    "db":        dict(label="Defensive backs", positions={"CB", "SS", "FS", "S", "DB"}, snap_col="defense_pct",
                      subgroup=lambda p: "CB" if p == "CB" else "S", measures="cutting", role="discovery"),
    "receivers": dict(label="Receivers", positions={"WR", "TE"}, snap_col="offense_pct",
                      subgroup=lambda p: "TE" if p == "TE" else "WR", measures="cutting", role="replication of DB"),
    "ol":        dict(label="Offensive line", positions={"T", "G", "C", "OT", "OG", "OL"}, snap_col="offense_pct",
                      subgroup=lambda p: "T" if p in {"T", "OT"} else "IOL", measures="burst", role="discovery"),
    "front7":    dict(label="Front seven", positions={"DE", "OLB", "DT", "NT"}, snap_col="defense_pct",
                      subgroup=lambda p: "EDGE" if p in {"DE", "OLB"} else "INT", measures="burst",
                      role="replication of OL"),
}
RUN = ["db", "receivers", "ol", "front7"]

# ---- pre-specified tests (findings doc, Nov 15, 2026) ----
# signature: load > LOAD_THR x baseline at game -2, then response drop > DROP_THR into game -1
# profile: injured vs healthy over games before onset, excluding the last 2; `predicted` = expected sign of Cliff's delta
TESTS = {
    "cutting": [
        dict(test="Load signature: cutting spike -> turn-rate drop", kind="signature",
             load="cuts_per_play", response="turn_rate", worse="down"),
        dict(test="Top-speed trend (weeks before injury)", kind="profile", metric="top_speed_trend", predicted=-1),
    ],
    "burst": [
        dict(test="Load signature: effort spike -> first-step drop", kind="signature",
             load="effort_per_rep", response="first_step", worse="down"),
        dict(test="First-step level", kind="profile", metric="first_step_level", predicted=-1),
        dict(test="Top-speed variability", kind="profile", metric="top_speed_variability", predicted=+1),
        dict(test="Combine weight", kind="profile", metric="combine_weight", predicted=-1),
    ],
}

# ---- shared settings (same values as the DB and OL notebooks) ----
MIN_BASELINE_GAMES, BASELINE_MIN_PCT, DROP_RATIO, MIN_OUT_GAMES, LONG_ABSENCE_GAMES = 3, 0.60, 0.50, 1, 4
MIN_SPEED_FOR_DIR, HARD_CUT_DEG_S, RETENTION_SEC = 2.0, 150, 0.5
GETOFF_YDS, GETOFF_MAX_SEC, FIRST_STEP_SEC, REP_SEC, ACC_CLIP = 1.0, 1.5, 0.8, 3.0, 12.0
MIN_PLAYS_PER_GAME = 5
PRE_WINDOW, MIN_BASE_GAMES_TRK, N_CONTROLS, CONTROL_SEED = 2, 2, 4, 0
LOAD_THR, DROP_THR = 1.10, 0.05
EXCLUDE_LAST, MIN_GAMES, PROFILE_SEED, MIN_PER_SEASON = 2, 4, 1, 3
MIN_CORR_N = 15
COMBINE_COLS = ["forty", "ten_yd_split", "vertical", "broad_jump", "three_cone", "short_shuttle", "bench_reps",
                "combine_weight"]
KEY_GAME = {"cutting": ["turn_rate", "retention", "top_speed"], "burst": ["get_off", "first_step", "stop_power"]}

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Synthetic demo mode" if USE_SYNTHETIC else f"Real data from {DATA_DIR.resolve()}")

In [ ]:
IDS = {"game_id": "gameId", "play_id": "playId", "nfl_id": "nflId"}
PP_COLS = ["game_id", "play_id", "nfl_id", "quarter", "play_direction", "pass_result"]


def load_nflverse(seasons):
    import nflreadpy as nfl
    return dict(snaps=nfl.load_snap_counts(seasons).to_pandas(),
                injuries=nfl.load_injuries(seasons).to_pandas(),
                id_map=nfl.load_players().to_pandas()[["gsis_id", "nfl_id", "pfr_id", "display_name", "position"]],
                schedule=nfl.load_schedules(seasons).to_pandas())


def tidy_tables(raw, nv):
    """Shared tables in the notebook's naming (camelCase IDs)."""
    T = {}
    games = raw["games"].rename(columns=IDS)
    T["games"] = games[games.season_type.astype(str).str.upper().str.startswith("REG")]
    T["seasons"] = sorted(T["games"].season.unique().tolist())
    T["players"] = raw["players"].rename(columns=IDS | {"nfl_position": "position", "display_name": "displayName"})
    pp = raw["player_play"].rename(columns=IDS)
    for c in ["quarter", "play_direction", "pass_result"]:
        if c not in pp.columns:
            pp[c] = np.nan
    T["plays"] = pp.drop_duplicates(["gameId", "playId"])[["gameId", "playId", "quarter", "play_direction"]]
    T["combine_res"] = raw["combine_results"].rename(columns=IDS).drop_duplicates("nflId", keep="last")
    T["snaps"], T["injuries"], T["schedule"] = nv["snaps"], nv["injuries"], nv["schedule"]
    idm = nv["id_map"].dropna(subset=["nfl_id"]).copy()
    idm["nflId"] = pd.to_numeric(idm.nfl_id, errors="coerce")
    T["id_map"] = idm.dropna(subset=["nflId"]).drop_duplicates("nflId")
    return T


def read_tracking(ids, chunksize=5_000_000):
    cols = ["game_id", "play_id", "nfl_id", "time", "x", "y", "s", "a", "dir", "event"]
    parts = []
    for f in sorted(glob.glob(str(DATA_DIR / TRACKING_GLOB))):
        for chunk in pd.read_csv(f, usecols=cols, chunksize=chunksize):
            parts.append(chunk[chunk.nfl_id.isin(ids)])
    return pd.concat(parts, ignore_index=True)


def read_combine_tracking(ids, chunksize=5_000_000):
    return pd.concat(c[c.nfl_id.isin(ids)] for c in pd.read_csv(DATA_DIR / "combine_tracking.csv", chunksize=chunksize))


if not USE_SYNTHETIC:
    RAW = {k: pd.read_csv(DATA_DIR / f"{k}.csv") for k in ["games", "players", "combine_results"]}
    RAW["player_play"] = pd.read_csv(DATA_DIR / "player_play.csv", usecols=lambda c: c in PP_COLS)
    NV = load_nflverse(sorted(RAW["games"].season.unique().tolist()))
    SHARED = tidy_tables(RAW, NV)
    print("seasons:", SHARED["seasons"])

## 1. Injury definition (same as the DB and OL notebooks)

A player with steady usage whose snap share drops below half his baseline and stays down (change point on the snap series), confirmed by the injury report. Concussions are excluded. Snap counts are matched by player ID, so each family uses its own players regardless of how nflverse labels positions.

In [ ]:
def build_snap_grid(snaps, schedule, pct_col):
    s = snaps[snaps.game_type == "REG"].copy()
    sch = schedule[schedule.game_type == "REG"]
    team_games = pd.concat([sch[["week", "home_team"]].rename(columns={"home_team": "team"}),
                            sch[["week", "away_team"]].rename(columns={"away_team": "team"})])
    weeks = np.sort(sch.week.unique())
    grid = (pd.MultiIndex.from_product([s.pfr_player_id.unique(), weeks], names=["pfr_player_id", "week"])
              .to_frame(index=False)
              .merge(s[["pfr_player_id", "week", "team", pct_col, "player", "position"]], how="left",
                     on=["pfr_player_id", "week"]))
    grid = grid.sort_values(["pfr_player_id", "week"])
    for c in ["team", "player", "position"]:
        grid[c] = grid.groupby("pfr_player_id")[c].transform(lambda x: x.ffill().bfill())
    grid = grid.merge(team_games, on=["week", "team"])
    grid["pct"] = grid[pct_col].fillna(0.0)
    return grid.sort_values(["pfr_player_id", "week"]).reset_index(drop=True)


def find_onset(pct, weeks):
    import ruptures as rpt
    if len(pct) < MIN_BASELINE_GAMES + 1:
        return None
    try:
        bkps = rpt.Pelt(model="l2", min_size=1, jump=1).fit(pct.reshape(-1, 1)).predict(pen=0.15)
    except Exception:
        bkps = [len(pct)]
    for i in sorted(set(b for b in bkps[:-1]) | set(b - 1 for b in bkps[:-1])):
        if i < MIN_BASELINE_GAMES:
            continue
        base = np.median(pct[i - MIN_BASELINE_GAMES:i])
        if base < BASELINE_MIN_PCT:
            continue
        if pct[i] < DROP_RATIO * base and pct[i:i + 1 + MIN_OUT_GAMES].mean() < DROP_RATIO * base:
            low = pct[i:] < DROP_RATIO * base
            return int(weeks[i]), float(base), int(np.argmin(low)) if not low.all() else int(low.sum())
    return None


def region(txt):
    t = str(txt).lower()
    if any(k in t for k in ["concussion", "head"]):
        return "concussion"
    for name, keys in [("leg", ["knee", "ankle", "foot", "hip", "toe", "acl", "mcl", "achilles", "hamstring",
                                "groin", "calf", "quad", "thigh", "shin", "leg"]),
                       ("upper", ["shoulder", "wrist", "elbow", "hand", "finger", "pectoral", "chest", "rib",
                                  "arm", "neck"]),
                       ("trunk", ["back", "abdomen", "oblique", "core"])]:
        if any(k in t for k in keys):
            return name
    return "other"


def identify_injuries(T, fam, fam_ids):
    pfr = set(T["id_map"].loc[T["id_map"].nflId.isin(fam_ids), "pfr_id"].dropna())
    snaps = T["snaps"][T["snaps"].pfr_player_id.isin(pfr)]
    grid = pd.concat([build_snap_grid(snaps[snaps.season == yr], T["schedule"][T["schedule"].season == yr],
                                      fam["snap_col"]).assign(season=yr) for yr in T["seasons"]], ignore_index=True)
    rows = []
    for (yr, pid), g in grid.groupby(["season", "pfr_player_id"]):
        r = find_onset(g.pct.to_numpy(), g.week.to_numpy())
        if r:
            rows.append(dict(season=yr, pfr_player_id=pid, player=g.player.iloc[0], onset_week=r[0],
                             baseline_pct=r[1], games_low=r[2]))
    onsets = pd.DataFrame(rows, columns=["season", "pfr_player_id", "player", "onset_week", "baseline_pct", "games_low"])
    onsets = onsets.merge(T["id_map"][["pfr_id", "gsis_id", "nflId"]], left_on="pfr_player_id", right_on="pfr_id",
                          how="left")
    rep = T["injuries"][T["injuries"].report_status.isin(["Out", "Doubtful", "Questionable"])]
    def match(row):
        m = rep[(rep.gsis_id == row.gsis_id) & (rep.season == row.season)
                & rep.week.between(row.onset_week, row.onset_week + 1)]
        return m.report_primary_injury.iloc[0] if len(m) else None
    onsets["reported_injury"] = onsets.apply(match, axis=1) if len(onsets) else []
    onsets["status"] = np.select([onsets.reported_injury.notna(), onsets.games_low >= LONG_ABSENCE_GAMES],
                                 ["confirmed", "likely_IR_unconfirmed"], "unconfirmed")
    onsets["region"] = np.where(onsets.reported_injury.notna(), onsets.reported_injury.map(region), None)
    tracked = T["games"].groupby("season").week.agg(["min", "max"])
    inj = onsets[(onsets.status == "confirmed") & (onsets.region != "concussion") & onsets.nflId.isin(fam_ids)].copy()
    inj = inj[[s in tracked.index and tracked.loc[s, "min"] < w <= tracked.loc[s, "max"] + 1
               for s, w in zip(inj.season, inj.onset_week)]]
    inj["nflId"] = inj.nflId.astype(int)
    pos_of = T["players"].set_index("nflId").position
    inj["subgroup"] = inj.nflId.map(pos_of).map(fam["subgroup"])
    return grid, onsets, inj

## 2. Game measures

Tracking is loaded per family, trimmed to start at the snap, and turned into rep-level measures, then game-level values. Rep-level results are cached so later runs skip the slow tracking load.

In [ ]:
GAME_KEYS = ["gameId", "playId", "nflId"]


def prep_tracking(trk, keys, snap_events=None):
    trk = trk.copy()
    trk["time"] = pd.to_datetime(trk.time, format="mixed")
    trk = trk.sort_values(keys + ["time"])
    if snap_events is not None and "event" in trk.columns:
        snap = trk[trk.event.isin(snap_events)].groupby(keys[:-1]).time.min().rename("snap_time")
        trk = trk.join(snap, on=keys[:-1])
        trk["has_snap"] = trk.snap_time.notna()
        trk = trk[~trk.has_snap | (trk.time >= trk.snap_time)].drop(columns="snap_time")
    trk["frameId"] = trk.groupby(keys).cumcount() + 1
    fps = float(round(1 / trk.groupby(keys).time.diff().dt.total_seconds().median()))
    return trk.reset_index(drop=True), fps


def cutting_reps(trk, fps):
    """DB-notebook measures per rep: peak speed, accel/decel, turn rate, hard cuts, speed kept through cuts."""
    trk = trk.sort_values(GAME_KEYS + ["frameId"]).copy()
    g = trk.groupby(GAME_KEYS, sort=False)
    w = max(1, int(round(RETENTION_SEC * fps)))
    trk["long_acc"] = (g.s.diff() * fps)
    trk["long_acc"] = trk.groupby(GAME_KEYS).long_acc.transform(
        lambda x: x.rolling(3, center=True, min_periods=1).mean()).clip(-ACC_CLIP, ACC_CLIP)
    unwrapped = g.dir.transform(lambda d: np.rad2deg(np.unwrap(np.deg2rad(d.to_numpy()))))
    trk["turn_rate"] = unwrapped.groupby([trk[k] for k in GAME_KEYS]).diff() * fps
    trk.loc[trk.s < MIN_SPEED_FOR_DIR, "turn_rate"] = np.nan
    trk["abs_turn"] = trk.groupby(GAME_KEYS).turn_rate.transform(
        lambda x: x.abs().rolling(3, center=True, min_periods=1).median())
    local_max = trk.groupby(GAME_KEYS).abs_turn.transform(lambda x: x.rolling(5, center=True, min_periods=1).max())
    trk["is_cut"] = (trk.abs_turn >= HARD_CUT_DEG_S) & (trk.abs_turn == local_max)
    s_b, s_a = g.s.shift(w), g.s.shift(-w)
    trk["retention"] = np.where(trk.is_cut & (s_b > MIN_SPEED_FOR_DIR), s_a / s_b, np.nan)
    rep = (trk.groupby(GAME_KEYS)
              .agg(max_s=("s", "max"), max_acc=("long_acc", "max"), max_decel=("long_acc", "min"),
                   max_turn=("abs_turn", "max"), retention=("retention", "mean"), n_cuts=("is_cut", "sum"))
              .reset_index())
    rep["max_decel"] = -rep.max_decel
    return rep


def burst_reps(trk, plays, fps):
    """OL-notebook measures per rep: get-off (interpolated), first step, stop power, top speed, peak accel."""
    trk = (trk[trk.has_snap].sort_values(GAME_KEYS + ["frameId"])
             .merge(plays[["gameId", "playId", "play_direction"]], on=["gameId", "playId"], how="left"))
    g = trk.groupby(GAME_KEYS, sort=False)
    trk["t"] = (trk.frameId - 1) / fps
    x0, y0 = g.x.transform("first"), g.y.transform("first")
    trk["disp"] = np.hypot(trk.x - x0, trk.y - y0)
    trk["long_acc"] = g.s.diff() * fps
    trk["long_acc"] = trk.groupby(GAME_KEYS).long_acc.transform(
        lambda v: v.rolling(3, center=True, min_periods=1).mean()).clip(-ACC_CLIP, ACC_CLIP)
    d_prev, t_prev = trk.groupby(GAME_KEYS).disp.shift(), trk.groupby(GAME_KEYS).t.shift()
    cross = (trk.disp >= GETOFF_YDS) & (d_prev < GETOFF_YDS) & (trk.t <= GETOFF_MAX_SEC)
    trk["go_t"] = np.where(cross, t_prev + (GETOFF_YDS - d_prev) / (trk.disp - d_prev) * (trk.t - t_prev), np.nan)
    trk["fs_acc"] = np.where(trk.t <= FIRST_STEP_SEC, trk.long_acc, np.nan)
    trk["acc_w"] = np.where(trk.t <= REP_SEC, trk.long_acc, np.nan)
    trk["s_w"] = np.where(trk.t <= REP_SEC, trk.s, np.nan)
    rep = (trk.groupby(GAME_KEYS)
              .agg(get_off=("go_t", "min"), first_step=("fs_acc", "max"), peak_acc=("acc_w", "max"),
                   stop_power=("acc_w", "min"), top_speed=("s_w", "max"))
              .reset_index())
    rep["stop_power"] = -rep.stop_power
    return rep


def game_level(rep, measures):
    q90, q75 = (lambda x: x.quantile(0.9)), (lambda x: x.quantile(0.75))
    if measures == "cutting":
        gl = rep.groupby(["nflId", "gameId"]).agg(
            top_speed=("max_s", q90), accel=("max_acc", q90), decel=("max_decel", q90), turn_rate=("max_turn", q90),
            retention=("retention", "median"), cuts_per_play=("n_cuts", "mean"), n_plays=("playId", "nunique"))
    else:
        gl = rep.groupby(["nflId", "gameId"]).agg(
            get_off=("get_off", "median"), first_step=("first_step", q75), stop_power=("stop_power", q90),
            top_speed=("top_speed", q90), effort_per_rep=("peak_acc", "median"), n_plays=("playId", "nunique"))
    return gl.reset_index()


FEATURES = {"cutting": ["top_speed", "accel", "decel", "turn_rate", "retention", "cuts_per_play"],
            "burst": ["get_off", "first_step", "stop_power", "top_speed", "effort_per_rep"]}


def family_reps(key, fam, T, fam_ids, raw=None):
    """Rep-level measures for a family, from cache when available."""
    cache = OUT_DIR / f"{key}_reps.parquet"
    if USE_CACHE and cache.exists():
        print(f"  reps from cache: {cache}")
        return pd.read_parquet(cache)
    trk = (raw["game_tracking"] if raw is not None else read_tracking(fam_ids)).rename(columns=IDS)
    trk = trk[trk.gameId.isin(T["games"].gameId)]
    trk, fps = prep_tracking(trk, GAME_KEYS, SNAP_EVENTS)
    print(f"  tracking: {len(trk):,} rows at {fps:.0f} Hz | tracked per play (median): "
          f"{trk.groupby(['gameId', 'playId']).nflId.nunique().median():.1f}")
    rep = cutting_reps(trk, fps) if fam["measures"] == "cutting" else burst_reps(trk, T["plays"], fps)
    if USE_CACHE:
        rep.to_parquet(cache)
    return rep

## 3. Pre-specified tests

**Controls:** up to 4 healthy players per injured player, same subgroup (e.g. WR vs. TE, edge vs. interior), similar snap share, same season and weeks. **Signature tests** compare the share of injured vs. control players showing the load-then-drop pattern (Fisher's exact test). **Profile tests** compare injured vs. healthy player-seasons over the games before onset, excluding the last 2 (Cliff's delta, Mann-Whitney), with a season check that only counts seasons with 3+ players per group.

In [ ]:
def cliffs_delta(a, b):
    a, b = np.asarray(pd.to_numeric(pd.Series(a), errors="coerce").dropna(), float), \
           np.asarray(pd.to_numeric(pd.Series(b), errors="coerce").dropna(), float)
    return (np.sign(a[:, None] - b[None, :])).mean() if len(a) and len(b) else np.nan


def mwu(a, b):
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna()
    b = pd.to_numeric(pd.Series(b), errors="coerce").dropna()
    return stats.mannwhitneyu(a, b).pvalue if len(a) > 1 and len(b) > 1 else np.nan


def match_controls(inj, gl, grid, T, fam, fam_ids, onsets):
    base_usage = (grid.merge(T["id_map"][["pfr_id", "nflId"]], left_on="pfr_player_id", right_on="pfr_id")
                      .groupby(["season", "nflId"]).pct.median())
    pos_of = T["players"].set_index("nflId").position
    ever = {(r.season, int(r.nflId)) for r in onsets.dropna(subset=["nflId"]).itertuples()}
    weeks = gl.groupby(["season", "nflId"]).week.apply(np.array).to_dict()
    rng = np.random.default_rng(CONTROL_SEED)
    rows = []
    for r in inj.itertuples():
        pool = [p for p in fam_ids
                if (r.season, p) not in ever and fam["subgroup"](pos_of.get(p)) == r.subgroup
                and abs(base_usage.get((r.season, p), -1) - r.baseline_pct) <= 0.15
                and (weeks.get((r.season, p), np.array([])) < r.onset_week).sum() >= MIN_BASE_GAMES_TRK + PRE_WINDOW]
        for c in rng.choice(pool, min(N_CONTROLS, len(pool)), replace=False):
            rows.append(dict(nflId=int(c), season=r.season, onset_week=r.onset_week, match_of=r.nflId))
    controls = pd.DataFrame(rows, columns=["nflId", "season", "onset_week", "match_of"])
    subjects = pd.concat([inj[["nflId", "season", "onset_week"]].assign(group="injured", match_of=inj.nflId),
                          controls.assign(group="control")], ignore_index=True)
    return subjects, base_usage, ever


def relative_series(gl, subjects, features):
    out = []
    for sid, s in enumerate(subjects.itertuples()):
        g = gl[(gl.nflId == s.nflId) & (gl.season == s.season)].sort_values("week").copy()
        pre = g[g.week < s.onset_week].copy()
        if len(pre) < MIN_BASE_GAMES_TRK + PRE_WINDOW:
            continue
        pre["rel_game"] = -np.arange(len(pre), 0, -1)
        base = pre.iloc[:-PRE_WINDOW][features].median()
        both = pd.concat([pre, g[g.week == s.onset_week].assign(rel_game=0)])
        both[features] = both[features] / base
        both["subject"], both["group"], both["season"] = sid, s.group, s.season
        out.append(both)
    cols = ["rel_game", "subject", "group", "season"] + features
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=cols)


def signature_test(rel, t):
    rows = []
    for (sid, grp_), g in rel.groupby(["subject", "group"]):
        s = g.set_index("rel_game")
        if not {-2, -1} <= set(s.index):
            continue
        drop = (s.loc[-2, t["response"]] - s.loc[-1, t["response"]]) * (1 if t["worse"] == "down" else -1)
        rows.append(dict(group=grp_, season=g.season.iloc[0], load_m2=s.loc[-2, t["load"]], drop=drop))
    sig = pd.DataFrame(rows, columns=["group", "season", "load_m2", "drop"])
    sig["signature"] = (sig.load_m2 > LOAD_THR) & (sig["drop"] > DROP_THR)
    a, b = sig[sig.group == "injured"], sig[sig.group == "control"]
    if len(a) and len(b):
        orr, p = stats.fisher_exact([[a.signature.sum(), (~a.signature).sum()], [b.signature.sum(), (~b.signature).sum()]])
    else:
        orr, p = np.nan, np.nan
    seasons = [(yr, g[g.group == "injured"].signature.mean() - g[g.group == "control"].signature.mean())
               for yr, g in sig.groupby("season")
               if (g.group == "injured").sum() >= MIN_PER_SEASON and (g.group == "control").sum() >= MIN_PER_SEASON]
    return dict(n_injured=len(a), n_control=len(b),
                injured_value=a.signature.mean() if len(a) else np.nan,
                control_value=b.signature.mean() if len(b) else np.nan,
                effect=orr, effect_type="odds ratio", p_value=p,
                direction_ok=bool(len(a) and len(b) and a.signature.mean() > b.signature.mean()),
                seasons_agree=f"{sum(d > 0 for _, d in seasons)}/{len(seasons)}")


def profile_table(gl, inj, base_usage, ever, T, fam, fam_ids, features):
    pos_of = T["players"].set_index("nflId").position
    inj_onset = {(r.season, r.nflId): r.onset_week for r in inj.itertuples()}
    rng = np.random.default_rng(PROFILE_SEED)
    windows = [(s, n, w, "injured") for (s, n), w in inj_onset.items()]
    if len(inj):
        for (s, n), usage in base_usage.items():
            if usage >= BASELINE_MIN_PCT and (s, int(n)) not in ever and n in fam_ids:
                windows.append((s, int(n), int(rng.choice(inj.onset_week)), "healthy"))
    rows = []
    for s, n, end, grp in windows:
        g = gl[(gl.nflId == n) & (gl.season == s) & (gl.week < end)].sort_values("week")
        g = g.iloc[:-EXCLUDE_LAST] if EXCLUDE_LAST else g
        if len(g) < MIN_GAMES:
            continue
        x = np.arange(len(g))
        r = dict(season=s, nflId=n, group=grp)
        for f in features:
            r[f"{f}_level"] = g[f].median()
            v = g[f] / g[f].median()
            r[f"{f}_variability"] = v.std()
            r[f"{f}_trend"] = np.polyfit(x[v.notna()], v.dropna(), 1)[0] if v.notna().sum() >= MIN_GAMES else np.nan
        rows.append(r)
    prof = pd.DataFrame(rows)
    if prof.empty:
        return prof
    prof["subgroup"] = prof.nflId.map(pos_of).map(fam["subgroup"])
    lv = [c for c in prof.columns if c.endswith("_level")]
    prof[lv] = prof.groupby(["season", "subgroup"])[lv].transform(lambda x: (x - x.mean()) / x.std())
    cc = [c for c in COMBINE_COLS if c in T["combine_res"].columns]
    prof = prof.merge(T["combine_res"][["nflId"] + cc], on="nflId", how="left")
    prof[cc] = prof.groupby("subgroup")[cc].transform(lambda x: (x - x.mean()) / x.std())
    return prof


def profile_test(prof, metric, predicted, seasons):
    if prof.empty or metric not in prof.columns:
        return dict(n_injured=0, n_control=0, injured_value=np.nan, control_value=np.nan, effect=np.nan,
                    effect_type="Cliff's delta", p_value=np.nan, direction_ok=False, seasons_agree="0/0")
    a, b = prof.loc[prof.group == "injured", metric].dropna(), prof.loc[prof.group == "healthy", metric].dropna()
    d = cliffs_delta(a, b)
    per = []
    for s in seasons:
        ai = prof.loc[(prof.group == "injured") & (prof.season == s), metric].dropna()
        bi = prof.loc[(prof.group == "healthy") & (prof.season == s), metric].dropna()
        if len(ai) >= MIN_PER_SEASON and len(bi) >= MIN_PER_SEASON:
            per.append(cliffs_delta(ai, bi))
    return dict(n_injured=len(a), n_control=len(b), injured_value=a.median(), control_value=b.median(),
                effect=d, effect_type="Cliff's delta", p_value=mwu(a, b),
                direction_ok=bool(np.isfinite(d) and np.sign(d) == predicted),
                seasons_agree=f"{sum(np.sign(v) == predicted for v in per)}/{len(per)}")

## 4. Combine comparison

Two questions per family, all within subgroup (z-scored within e.g. WR vs. TE, so position differences don't create false correlations):
1. **Do Combine results separate injured from healthy players?** Largest |Cliff's delta| across Combine results, and how many reach p < 0.05.
2. **Do tracked drills match game movement better than drill times?** For each key game measure, the best |ρ| with any Combine result vs. the best |ρ| with any tracked drill measure (min. 15 players).

In [ ]:
COMBINE_KEYS = ["nflId", "event_id", "drill_name", "attempt"]


def combine_attempt_features(ct, fps):
    ct = ct.sort_values(COMBINE_KEYS + ["frameId"]).copy()
    moving = ct.s >= 0.3
    ct["start_t"] = ct.time.where(moving).groupby([ct[k] for k in COMBINE_KEYS]).transform("min")
    ct["t"] = (ct.time - ct.start_t).dt.total_seconds()
    ct = ct[ct.t >= 0].copy()
    g = ct.groupby(COMBINE_KEYS, sort=False)
    x0, y0 = g.x.transform("first"), g.y.transform("first")
    ct["disp"] = np.hypot(ct.x - x0, ct.y - y0)
    ct["long_acc"] = (g.s.diff() * fps)
    ct["long_acc"] = ct.groupby(COMBINE_KEYS).long_acc.transform(
        lambda v: v.rolling(3, center=True, min_periods=1).mean()).clip(-ACC_CLIP, ACC_CLIP)
    unwrapped = g.dir.transform(lambda d: np.rad2deg(np.unwrap(np.deg2rad(d.to_numpy()))))
    ct["turn"] = unwrapped.groupby([ct[k] for k in COMBINE_KEYS]).diff().abs() * fps
    ct.loc[ct.s < MIN_SPEED_FOR_DIR, "turn"] = np.nan
    w = max(1, int(round(RETENTION_SEC * fps)))
    is_cut = (ct.turn >= HARD_CUT_DEG_S) & (ct.turn == ct.groupby(COMBINE_KEYS).turn.transform(
        lambda v: v.rolling(5, center=True, min_periods=1).max()))
    s_b, s_a = g.s.shift(w), g.s.shift(-w)
    ct["retention"] = np.where(is_cut & (s_b > MIN_SPEED_FOR_DIR), s_a / s_b, np.nan)
    ct["fs"] = np.where(ct.t <= FIRST_STEP_SEC, ct.long_acc, np.nan)
    ct["t1"] = np.where(ct.disp >= 1, ct.t, np.nan)
    ct["t10"] = np.where(ct.disp >= 10, ct.t, np.nan)
    out = (ct.groupby(COMBINE_KEYS)
             .agg(first_step=("fs", "max"), time_1yd=("t1", "min"), time_10yd=("t10", "min"),
                  max_decel=("long_acc", "min"), max_turn=("turn", "max"), retention=("retention", "mean"))
             .reset_index())
    out["max_decel"] = -out.max_decel
    return out


def family_drills(key, fam_ids, raw=None):
    cache = OUT_DIR / f"{key}_combine_attempts.parquet"
    if USE_CACHE and cache.exists():
        return pd.read_parquet(cache)
    ct = (raw["combine_tracking"] if raw is not None else read_combine_tracking(fam_ids)).rename(columns=IDS)
    ct = ct[ct.nflId.isin(fam_ids)].dropna(subset=["nflId"]).copy()
    if ct.empty:
        return pd.DataFrame(columns=COMBINE_KEYS)
    ct["drill_name"] = (ct.drill_name.astype(str).str.upper().str.replace("-", "_", regex=False)
                          .str.replace(r"_(LEFT|RIGHT)$", "", regex=True))
    ct, fps = prep_tracking(ct, COMBINE_KEYS)
    att = combine_attempt_features(ct, fps)
    if USE_CACHE:
        att.to_parquet(cache)
    return att


def combine_comparison(gl, prof, attempts, T, fam, measures):
    rows = []
    # (1) Combine results: injured vs healthy (profile table already has them z-scored within subgroup)
    cc = [c for c in COMBINE_COLS if c in prof.columns] if not prof.empty else []
    seps = []
    for c in cc:
        a = prof.loc[prof.group == "injured", c].dropna(); b = prof.loc[prof.group == "healthy", c].dropna()
        if len(a) >= 5 and len(b) >= 5:
            seps.append((c, cliffs_delta(a, b), mwu(a, b)))
    best = max(seps, key=lambda t: abs(t[1])) if seps else (None, np.nan, np.nan)
    sep = dict(best_combine_result=best[0], best_abs_delta=abs(best[1]) if best[0] else np.nan,
               best_p=best[2], n_results_p05=sum(p < 0.05 for _, _, p in seps), n_results_tested=len(seps))

    # (2) drill vs game, within subgroup
    pos_of = T["players"].set_index("nflId").position
    key_game = KEY_GAME[measures]
    pt = gl.groupby("nflId")[key_game].median().reset_index()
    pt["subgroup"] = pt.nflId.map(pos_of).map(fam["subgroup"])
    res_cols = [c for c in COMBINE_COLS if c in T["combine_res"].columns]
    pt = pt.merge(T["combine_res"][["nflId"] + res_cols], on="nflId", how="left")
    drill_cols = []
    if len(attempts):
        dp = attempts.groupby(["nflId", "drill_name"])[["first_step", "time_1yd", "time_10yd", "max_decel",
                                                          "max_turn", "retention"]].median().unstack("drill_name")
        dp.columns = [f"{d}|{f}" for f, d in dp.columns]
        dp = dp.dropna(axis=1, thresh=MIN_CORR_N).reset_index()
        drill_cols = [c for c in dp.columns if c != "nflId"]
        pt = pt.merge(dp, on="nflId", how="left")
    zc = key_game + res_cols + drill_cols
    pt[zc] = pt.groupby("subgroup")[zc].transform(lambda x: (x - x.mean()) / x.std())

    def best_corr(cols, target):
        best_c, best_r, best_n = None, np.nan, 0
        for c in cols:
            d = pt[[c, target]].dropna()
            if len(d) < MIN_CORR_N:
                continue
            r = d[c].corr(d[target], method="spearman")
            if np.isfinite(r) and (best_c is None or abs(r) > abs(best_r)):
                best_c, best_r, best_n = c, r, len(d)
        return best_c, best_r, best_n

    for m in key_game:
        rc, rr, rn = best_corr(res_cols, m)
        dc, dr, dn = best_corr(drill_cols, m)
        rows.append(dict(game_measure=m, best_result=rc, result_rho=rr, result_n=rn,
                         best_tracked=dc, tracked_rho=dr, tracked_n=dn,
                         tracked_better=bool(np.isfinite(dr) and (not np.isfinite(rr) or abs(dr) > abs(rr)))))
    return sep, pd.DataFrame(rows)

## 5. Run all families

Each family prints its injury counts and data checks as it goes. Results land in `RESULTS[key]`.

In [ ]:
def run_family(key):
    fam = FAMILIES[key]
    print(f"\n=== {fam['label']} ({key}) ===")
    raw = None
    if USE_SYNTHETIC:
        from synthetic_cross import make_family
        raw = make_family(key)
        T = tidy_tables(raw, raw)
    else:
        T = SHARED
    fam_ids = set(T["players"].loc[T["players"].position.isin(fam["positions"]), "nflId"])
    print(f"  players: {len(fam_ids)} | positions: "
          f"{T['players'].loc[T['players'].nflId.isin(fam_ids), 'position'].value_counts().to_dict()}")

    grid, onsets, inj = identify_injuries(T, fam, fam_ids)
    print(f"  onsets: {onsets.status.value_counts().to_dict()} | confirmed in tracking window: {len(inj)} "
          f"| regions: {inj.region.value_counts().to_dict()}")

    rep = family_reps(key, fam, T, fam_ids, raw)
    gl = game_level(rep, fam["measures"]).merge(T["games"][["gameId", "season", "week"]], on="gameId")
    gl = gl[gl.n_plays >= MIN_PLAYS_PER_GAME]
    print(f"  player-games with >= {MIN_PLAYS_PER_GAME} reps: {len(gl):,}")

    feats = FEATURES[fam["measures"]]
    subjects, base_usage, ever = match_controls(inj, gl, grid, T, fam, fam_ids, onsets)
    rel = relative_series(gl, subjects, feats)
    prof = profile_table(gl, inj, base_usage, ever, T, fam, fam_ids, feats)
    print(f"  subjects: {subjects.group.value_counts().to_dict()} | profile player-seasons: "
          f"{prof.group.value_counts().to_dict() if not prof.empty else {}}")

    tests = []
    for t in TESTS[fam["measures"]]:
        r = signature_test(rel, t) if t["kind"] == "signature" else profile_test(prof, t["metric"], t["predicted"],
                                                                               T["seasons"])
        tests.append(dict(family=fam["label"], role=fam["role"], test=t["test"]) | r)

    attempts = family_drills(key, fam_ids, raw)
    sep, drill_vs_game = combine_comparison(gl, prof, attempts, T, fam, fam["measures"])
    return dict(fam=fam, T=T, inj=inj, onsets=onsets, gl=gl, rel=rel, prof=prof, subjects=subjects,
                tests=pd.DataFrame(tests), combine_sep=sep, drill_vs_game=drill_vs_game.assign(family=fam["label"]))


RESULTS = {}
for key in RUN:
    RESULTS[key] = run_family(key)

## 6. Cross-position summary

**Table A** is the main result: every pre-specified test, in every family. Discovery rows reproduce what the DB and OL notebooks found (so they will look good by construction); **replication rows are the real tests.** `direction_ok` = the effect points the predicted way; `seasons_agree` = seasons (with 3+ players per group) pointing the predicted way.

In [ ]:
A = pd.concat([r["tests"] for r in RESULTS.values()], ignore_index=True)
A["p_value"] = A.p_value.astype(float)
display(A[["family", "role", "test", "n_injured", "n_control", "injured_value", "control_value", "effect",
           "effect_type", "p_value", "direction_ok", "seasons_agree"]].round(3))

**Table B** — Combine results vs. injury, per family. Do any Combine results separate injured from healthy players?

In [ ]:
B = pd.DataFrame([dict(family=r["fam"]["label"]) | r["combine_sep"] for r in RESULTS.values()])
display(B.round(3))

**Table C** — Do tracked drills match game movement better than drill times? One row per family and key game measure, all within subgroup.

In [ ]:
C = pd.concat([r["drill_vs_game"] for r in RESULTS.values()], ignore_index=True)
display(C[["family", "game_measure", "best_result", "result_rho", "result_n", "best_tracked", "tracked_rho",
           "tracked_n", "tracked_better"]].round(2))

In [ ]:
# One figure: replication vs discovery at a glance (profile tests: Cliff's delta; signature tests: rate difference)
plot_rows = A.copy()
plot_rows["value"] = np.where(plot_rows.effect_type == "odds ratio", plot_rows.injured_value - plot_rows.control_value,
                              plot_rows.effect)
plot_rows["label"] = plot_rows.family + " — " + plot_rows.test
fig, ax = plt.subplots(figsize=(9, 0.45 * len(plot_rows) + 1))
colors = ["crimson" if r.startswith("replication") else "grey" for r in plot_rows.role]
ax.barh(plot_rows.label, plot_rows.value, color=colors)
ax.axvline(0, color="k", lw=0.8)
ax.invert_yaxis()
ax.set_xlabel("Cliff's delta (profile tests) or injured − control signature rate (signature tests)")
ax.set_title("Pre-specified tests by family (red = replication, grey = discovery)")
plt.tight_layout(); plt.show()

In [ ]:
A.to_csv(OUT_DIR / "table_a_prespecified_tests.csv", index=False)
B.to_csv(OUT_DIR / "table_b_combine_vs_injury.csv", index=False)
C.to_csv(OUT_DIR / "table_c_drills_vs_game.csv", index=False)
print("saved to", OUT_DIR.resolve())

### Reading the results

- **Replication rows decide the generality claim.** If the receiver and front-seven rows point the predicted way (and agree across seasons), the DB and OL findings extend beyond those groups. If they don't, report that honestly: the injury patterns are position-specific, and the Combine argument rests on the shared finding below.
- **The shared finding to check across all four families:** Combine results don't separate injured players (Table B), and tracked drills match game movement about as well as, or better than, times (Table C). That is the Combine argument, and it can hold even where the specific injury patterns differ.
- **Caveats:** small numbers of injured players per family; injury defined from snaps and injury reports; partial tracking per play; RBs, QBs, and inside linebackers are effectively untracked and out of scope.